# HieraCascade-STS Demo

This notebook demonstrates the complete HieraCascade-STS pipeline for soft-tissue tumor classification.

## Architecture Overview

**Stage 1**: Full volume (192³) → Coarse predictions + Saliency map  
**Stage 2**: K=8 high-res crops (96³) → Hierarchical fine+coarse classification

### Key Features
- ✅ Mask-free (no segmentation required)
- ✅ Multi-modal (CT and MRI)
- ✅ Hierarchical (fine → coarse class relationships)
- ✅ Efficient (cascaded architecture)

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from hieracascade.dataio import (
    create_index_from_sheet,
    create_site_held_out_splits,
    DatasetStage1,
    FINE_TO_IDX,
    COARSE_TO_IDX
)

%matplotlib inline

## 1. Load Data from sheet.csv

In [ ]:
# Load dataset index from sheet.csv
index = create_index_from_sheet(
    data_root='../data',
    sheet_path='../data/sheet.csv',
    output_csv='../data/labels_hieracascade.csv'
)

print(f"\nLoaded {len(index)} studies")
print(f"\nFirst sample:")
print(index[0])

## 2. Create Train/Val Splits

In [ ]:
# Site-held-out cross-validation
splits = create_site_held_out_splits(index)
train_index, val_index = splits[0]  # Fold 0

print(f"Train: {len(train_index)} studies")
print(f"Val: {len(val_index)} studies")

## 3. Visualize Sample Volume

In [ ]:
from hieracascade.dataio import preprocess_volume

# Load and preprocess a sample
sample = train_index[0]
volume = preprocess_volume(
    sample['path'],
    sample['modality'],
    target_spacing=1.5,
    target_size=(192, 192, 192)
)

print(f"Study: {sample['study_id']}")
print(f"Category: {sample['category']}")
print(f"Modality: {sample['modality']}")
print(f"Volume shape: {volume.shape}")

# Visualize mid-slices
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(volume[96, :, :], cmap='gray')
axes[0].set_title('Axial')
axes[0].axis('off')

axes[1].imshow(volume[:, 96, :], cmap='gray')
axes[1].set_title('Coronal')
axes[1].axis('off')

axes[2].imshow(volume[:, :, 96], cmap='gray')
axes[2].set_title('Sagittal')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 4. Build Stage-1 Model

In [ ]:
from hieracascade.models import build_stage1_model

# Build Stage-1 model
model_stage1 = build_stage1_model(
    n_classes=len(FINE_TO_IDX),
    backbone='swin3d_t',
    embed_dim=96
)

print(f"Stage-1 parameters: {model_stage1.get_num_params():,}")

# Test forward pass
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_stage1 = model_stage1.to(device)

# Dummy input
x = torch.randn(1, 1, 192, 192, 192).to(device)
modality_id = torch.tensor([0]).to(device)  # 0 for CT

with torch.no_grad():
    logits, saliency = model_stage1(x, modality_id)

print(f"Logits shape: {logits.shape}")
print(f"Saliency shape: {saliency.shape}")

## 5. Build Stage-2 Model

In [ ]:
from hieracascade.models import build_stage2_model

# Build Stage-2 model
model_stage2 = build_stage2_model(
    n_fine=len(FINE_TO_IDX),
    n_coarse=len(COARSE_TO_IDX),
    backbone='swin3d_b',
    embed_dim=768,
    pooling='set_transformer'
)

print(f"Stage-2 parameters: {model_stage2.get_num_params():,}")

# Test forward pass with crops
model_stage2 = model_stage2.to(device)

# Dummy crops (B=1, K=8, 1, 96, 96, 96)
crops = torch.randn(1, 8, 1, 96, 96, 96).to(device)
modality_id = torch.tensor([0]).to(device)

with torch.no_grad():
    logits_fine, logits_coarse, embedding = model_stage2(crops, modality_id)

print(f"Fine logits shape: {logits_fine.shape}")
print(f"Coarse logits shape: {logits_coarse.shape}")
print(f"Study embedding shape: {embedding.shape}")

## 6. Training

For training, use the command-line scripts:

```bash
# Quick start (both stages)
python -m hieracascade.quick_start \
    --data_root data \
    --sheet_csv data/sheet.csv \
    --output_dir outputs/hieracascade \
    --fold 0

# Or train stages separately:

# Stage-1
python -m hieracascade.train_stage1 \
    --config hieracascade/configs/stage1.yaml \
    --data_root data \
    --labels_csv data/labels_hieracascade.csv \
    --output_dir outputs/stage1/fold0 \
    --fold 0

# Stage-2
python -m hieracascade.train_stage2 \
    --config hieracascade/configs/stage2.yaml \
    --data_root data \
    --labels_csv data/labels_hieracascade.csv \
    --stage1_ckpt outputs/stage1/fold0/checkpoint_best.pt \
    --output_dir outputs/stage2/fold0 \
    --fold 0
```

## 7. Load and Evaluate Trained Model

After training, load checkpoints and evaluate:

In [ ]:
# Example: Load trained checkpoint
# checkpoint_path = 'outputs/stage2/fold0/checkpoint_best.pt'
# checkpoint = torch.load(checkpoint_path)
# model_stage2.load_state_dict(checkpoint['model_state_dict'])
# print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
# print(f"Val metrics: {checkpoint['metrics']}")

## 8. Inference Example

In [ ]:
from hieracascade.dataio import get_crop_centers_from_saliency, extract_crop

# Inference pipeline on a single volume
@torch.no_grad()
def predict_volume(volume_path, modality, model_s1, model_s2, device='cuda'):
    """
    Complete inference pipeline
    
    Args:
        volume_path: Path to NIfTI volume
        modality: 'CT' or 'MRI'
        model_s1: Stage-1 model
        model_s2: Stage-2 model
        device: Device
    
    Returns:
        predictions dict with fine/coarse predictions
    """
    # 1. Preprocess
    volume = preprocess_volume(
        volume_path, modality,
        target_spacing=1.5,
        target_size=(192, 192, 192)
    )
    
    volume_tensor = torch.from_numpy(volume[None, None]).float().to(device)
    modality_id = torch.tensor([0 if modality == 'CT' else 1]).to(device)
    
    # 2. Stage-1: Get saliency
    model_s1.eval()
    logits1, saliency = model_s1(volume_tensor, modality_id)
    
    # 3. Extract crops from saliency
    centers = get_crop_centers_from_saliency(
        saliency[0, 0], K=8, nms_distance=16
    )
    
    crops = []
    for center in centers:
        crop = extract_crop(volume, center, crop_size=96)
        crops.append(crop)
    
    crops_tensor = torch.from_numpy(
        np.stack(crops)[:, None]
    ).float().to(device)
    crops_tensor = crops_tensor.unsqueeze(0)  # (1, K, 1, 96, 96, 96)
    
    # 4. Stage-2: Final predictions
    model_s2.eval()
    logits_fine, logits_coarse, _ = model_s2(crops_tensor, modality_id)
    
    # 5. Get predictions
    prob_fine = torch.softmax(logits_fine, dim=1)[0]
    prob_coarse = torch.softmax(logits_coarse, dim=1)[0]
    
    pred_fine = torch.argmax(prob_fine).item()
    pred_coarse = torch.argmax(prob_coarse).item()
    
    fine_classes = list(FINE_TO_IDX.keys())
    coarse_classes = list(COARSE_TO_IDX.keys())
    
    return {
        'fine_class': fine_classes[pred_fine],
        'fine_confidence': prob_fine[pred_fine].item(),
        'coarse_class': coarse_classes[pred_coarse],
        'coarse_confidence': prob_coarse[pred_coarse].item(),
        'saliency': saliency[0, 0].cpu().numpy(),
        'crop_centers': centers
    }

# Example usage (after training):
# result = predict_volume(
#     sample['path'],
#     sample['modality'],
#     model_stage1,
#     model_stage2,
#     device=device
# )
# print(f"Predicted: {result['fine_class']} (conf: {result['fine_confidence']:.3f})")
# print(f"Family: {result['coarse_class']} (conf: {result['coarse_confidence']:.3f})")

## 9. Visualize Results

After training and evaluation, results are saved to:

- `outputs/stage1/fold0/visualizations/` - Saliency overlays
- `outputs/stage2/fold0/plots/` - Training curves
- `outputs/stage2/fold0/eval/` - Confusion matrices, predictions CSV

Load and display:

In [ ]:
from IPython.display import Image, display

# Example: Display training curves
# display(Image('outputs/stage2/fold0/plots/training_curves.png'))

# Example: Display confusion matrix
# display(Image('outputs/stage2/fold0/eval/confusion_matrix_fine.png'))

## Summary

This notebook demonstrated:
1. ✅ Loading data from sheet.csv
2. ✅ Building Stage-1 and Stage-2 models
3. ✅ Training pipeline overview
4. ✅ Inference on new volumes

For production training, use the command-line scripts for better GPU utilization and checkpointing.